# 쉐보레 홈페이지 FAQ 크롤링 가이드 (Playwright)

## 개요

쉐보레 홈페이지의 FAQ 페이지는 **TOP 10 탭이 기본 선택 상태**이며,  
질문을 클릭하면 답변이 펼쳐지는 **아코디언(Accordion) 구조**로 구성되어 있습니다.

본 문서는 Playwright를 활용하여 다음 데이터를 수집하는 방법을 정리합니다.

- `TOP 10` 탭을 명확히 선택
- `[차량관리]`, `[기타]` 카테고리 질문 식별
- 질문 클릭을 통한 답변 전개
- 펼쳐진 답변 텍스트 추출

In [1]:
# from playwright.sync_api import sync_playwright       # 동기 방식 크롤링
from playwright.async_api import async_playwright       # 비동기 방식 크롤링

import time
import json

import asyncio
from urllib.parse import urljoin

In [2]:
async def check_connection():
    target_url = "https://www.chevrolet.co.kr/faq"

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False) # 코랩은 True 필수, 로컬은 False로 테스트
        
        # 봇 탐지 우회를 위해 평범한 User-Agent 추가
        context = await browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        print(f"접속 시도 중: {target_url}")
        
        try:
            # networkidle 대신 domcontentloaded 사용, 타임아웃 60초로 넉넉하게 설정
            await page.goto(target_url, wait_until="domcontentloaded", timeout=60000)
            print("페이지 기본 로드 성공!")
            
            # 특정 핵심 요소가 화면에 나타날 때까지 명시적으로 대기 (TOP 10 텍스트)
            await page.wait_for_selector("text=TOP 10", timeout=15000)
            print("FAQ 요소 로드 확인 완료. 크롤링을 진행할 수 있습니다.")
            
        except Exception as e:
            print(f"접속 실패 에러 상세: {e}")
            
        finally:
            await browser.close()

In [3]:
# 주피터 노트북 실행
await check_connection()

접속 시도 중: https://www.chevrolet.co.kr/faq
페이지 기본 로드 성공!
FAQ 요소 로드 확인 완료. 크롤링을 진행할 수 있습니다.


In [ ]:
async def crawl_chevrolet_faq():
    base_url = "https://www.chevrolet.co.kr"
    target_url = "https://www.chevrolet.co.kr/faq"
    all_data = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
            )
        page = await context.new_page()

        print(f"접속 중: {target_url}")
        await page.goto(target_url, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(3000)
        print("페이지 로드 완료. 수집을 시작합니다.")
        
        # 1. 탭 이름과 링크(href) 사전 추출
        tab_selectors = page.locator("main#gb-main-content adv-grid.hide-for-small a.q-link")
        tab_count = await tab_selectors.count()
        tabs_info = []
        
        for i in range(tab_count):
            tab = tab_selectors.nth(i)
            name = (await tab.inner_text()).strip()
            href = await tab.get_attribute("href")
            
            # TOP 10 제외 및 유효한 링크만 수집
            if name and name != "TOP 10" and href:
                # urljoin을 통해 전체 URL 생성 (예: https://www.chevrolet.co.kr/faq/purchasing-related)
                full_url = urljoin(base_url, href)
                tabs_info.append({"name": name, "url": full_url})
                
        print(f"총 {len(tabs_info)}개의 탭 링크 추출 완료 (TOP 10 제외)")

        # 2. 수집한 URL로 직접 이동하며 순회
        for tab in tabs_info:
            tab_name = tab["name"]
            tab_url = tab["url"]

            print(f"\n--- [{tab_name}] 탭 수집 시작 ---")
            print(tab_url)

            # 불안정한 click() 대신 URL 직접 접속
            await page.goto(tab_url, wait_until="domcontentloaded", timeout=60000)
            await page.wait_for_timeout(3000)

            expanders = page.locator("main#gb-main-content gb-adv-grid.gb-large-margin gb-expander")
            #gb-main-content > gb-adv-grid.gb-large-margin > adv-col > div > gb-expander.gb-expander.none-margin.active > div.gb-expander-content > div > gb-content-well > gb-adv-grid > adv-col > div > div > div > p
            try:
                await expanders.first.wait_for(state="attached", timeout=10000)
            except:
                print("  -> 조건(gb-adv-grid.large-margin)에 맞는 질문이 없어 다음 탭으로 넘어갑니다.")
                continue
                
            expander_count = await expanders.count()
            print(f"  -> {expander_count}개의 타겟 질문 발견")

            # 3. 질문 및 답변 추출
            for j in range(expander_count):
                try:
                    expander = expanders.nth(j)
                    question_text = await expander.get_attribute("open-headline")
                    
                    if not question_text: continue

                    btn = expander.locator(".gb-expander-btn")
                    await btn.click(force=True)
                    await page.wait_for_timeout(300)

                    content_box = expander.locator(".gb-expander-content")
                    answer_text = (await content_box.inner_text()).strip()

                    all_data.append({
                        "category": tab_name,
                        "question": question_text,
                        "answer": answer_text
                    })

                except Exception:
                    pass

        print(f"\n총 {len(all_data)}개의 FAQ 데이터를 수집했습니다.")
        
        with open("data/FAQ/chevrolet_FAQ_data.json", "w", encoding="utf-8") as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)

        await browser.close()
        return all_data


In [ ]:
# 주피터 노트북 실행부
result_data = await crawl_chevrolet_faq()

접속 중: https://www.chevrolet.co.kr/faq
페이지 로드 완료. 수집을 시작합니다.
총 10개의 탭 링크 추출 완료 (TOP 10 제외)

--- [구매 관련] 탭 수집 시작 ---
https://www.chevrolet.co.kr/faq/purchasing-related
  -> 24개의 타겟 질문 발견

--- [차량 관리] 탭 수집 시작 ---
https://www.chevrolet.co.kr/faq/product-maintenance
  -> 46개의 타겟 질문 발견

--- [오토카드] 탭 수집 시작 ---
https://www.chevrolet.co.kr/faq/autocard
  -> 4개의 타겟 질문 발견

--- [통합계정 및 홈페이지 이용] 탭 수집 시작 ---
https://www.chevrolet.co.kr/faq/website
  -> 13개의 타겟 질문 발견

--- [장애인 차량] 탭 수집 시작 ---
https://www.chevrolet.co.kr/faq/disabled-vehicles
  -> 9개의 타겟 질문 발견

--- [마이링크] 탭 수집 시작 ---
https://www.chevrolet.co.kr/faq/mylink
  -> 조건(gb-adv-grid.large-margin)에 맞는 질문이 없어 다음 탭으로 넘어갑니다.

--- [내비 업데이트] 탭 수집 시작 ---
https://www.chevrolet.co.kr/faq/navigation
  -> 5개의 타겟 질문 발견

--- [Android Auto/
Apple CarPlay] 탭 수집 시작 ---
https://www.chevrolet.co.kr/faq/android-auto-apple-carplay
  -> 2개의 타겟 질문 발견

--- [Online Shop] 탭 수집 시작 ---
https://www.chevrolet.co.kr/faq/online-shop
  -> 21개의 타겟 질문 발견

--- [EV 리콜] 탭 수집 시작 

In [ ]:
async def scrape_chevrolet_faq_mylink():
    target_url = "https://www.chevrolet.co.kr/faq/mylink"
    all_data = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False) # 로컬은 False로 해야 창을 로드하니 필수
        context = await browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
        page = await context.new_page()

        print(f"접속 중: {target_url}")
        await page.goto(target_url, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(3000) # 페이지 렌더링 대기
        print("페이지 로드 완료. 수집을 시작합니다.")

        print("\n--- [MyLink] 탭 수집 시작 ---")
        expanders = page.locator("main#gb-main-content gb-adv-grid.gb-small-margin gb-expander")
        
        try:
            await expanders.first.wait_for(state="attached", timeout=10000)
        except:
            print("조건(gb-small-margin)에 맞는 질문을 찾지 못했습니다.")
            await browser.close()
            return []
            
        expander_count = await expanders.count()
        print(f"  -> {expander_count}개의 타겟 질문 발견")

        # 질문 및 답변 추출
        for j in range(expander_count):
            try:
                expander = expanders.nth(j)
                
                # 질문 추출
                question_text = await expander.get_attribute("open-headline")
                if not question_text: continue

                # 아코디언 열기
                btn = expander.locator(".gb-expander-btn")
                await btn.click(force=True)
                await page.wait_for_timeout(300) # 애니메이션 대기

                # 답변 추출 (p 태그 텍스트 포함)
                content_box = expander.locator(".gb-expander-content")
                answer_text = (await content_box.inner_text()).strip()

                all_data.append({
                    "category": "MyLink",
                    "question": question_text,
                    "answer": answer_text
                })

            except Exception as e:
                pass # 개별 항목 에러 무시

        print(f"\n총 {len(all_data)}개의 MyLink FAQ 데이터를 수집했습니다.")
        
        with open("chevrolet_faq_mylink.json", "w", encoding="utf-8") as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)

        await browser.close()
        return all_data

# 주피터 노트북 실행부
result_data_mylink = await scrape_chevrolet_faq_mylink()

접속 중: https://www.chevrolet.co.kr/faq/mylink
페이지 로드 완료. 수집을 시작합니다.

--- [MyLink] 탭 수집 시작 ---
  -> 9개의 타겟 질문 발견

총 9개의 MyLink FAQ 데이터를 수집했습니다.


In [10]:
async def scrape_chevrolet_faq_mylink():
    target_url = "https://www.chevrolet.co.kr/faq/mylink"
    all_data = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False) # 로컬은 False로 해야 창을 로드하니 필수
        context = await browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
        page = await context.new_page()

        print(f"접속 중: {target_url}")
        await page.goto(target_url, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(3000) # 페이지 렌더링 대기
        print("페이지 로드 완료. 수집을 시작합니다.")

        print("\n--- [MyLink] 탭 수집 시작 ---")
        expanders = page.locator("main#gb-main-content adv-grid.large-margin gb-expander")
        
        try:
            await expanders.first.wait_for(state="attached", timeout=10000)
        except:
            print("조건(gb-small-margin)에 맞는 질문을 찾지 못했습니다.")
            await browser.close()
            return []
            
        expander_count = await expanders.count()
        print(f"  -> {expander_count}개의 타겟 질문 발견")

        # 질문 및 답변 추출
        for j in range(expander_count):
            try:
                expander = expanders.nth(j)
                
                # 질문 추출
                question_text = await expander.get_attribute("open-headline")
                if not question_text: continue

                # 아코디언 열기
                btn = expander.locator(".gb-expander-btn")
                await btn.click(force=True)
                await page.wait_for_timeout(300) # 애니메이션 대기

                # 답변 추출 (p 태그 텍스트 포함)
                content_box = expander.locator(".gb-expander-content")
                answer_text = (await content_box.inner_text()).strip()

                all_data.append({
                    "category": "MyLink",
                    "question": question_text,
                    "answer": answer_text
                })

            except Exception as e:
                pass # 개별 항목 에러 무시

        print(f"\n총 {len(all_data)}개의 MyLink FAQ 데이터를 수집했습니다.")
        
        with open("chevrolet_faq_mylink.json", "w", encoding="utf-8") as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)

        await browser.close()
        return all_data

# 주피터 노트북 실행부
result_data_mylink = await scrape_chevrolet_faq_mylink()

접속 중: https://www.chevrolet.co.kr/faq/mylink
페이지 로드 완료. 수집을 시작합니다.

--- [MyLink] 탭 수집 시작 ---
  -> 4개의 타겟 질문 발견

총 4개의 MyLink FAQ 데이터를 수집했습니다.


# Playwright 기반 쉐보레 FAQ 크롤링 이슈 해결 보고서

---

## 1. 초기 환경 설정 및 런타임 에러 해결

### 문제 1: Jupyter Notebook 환경에서의 동기식 코드 충돌  
**현상:** Error: It looks like you are using Playwright Sync API inside the asyncio loop. 에러 발생.  

**원인:** 주피터 노트북(또는 Colab)은 기본적으로 백그라운드에서 비동기(asyncio) 이벤트 루프가 실행 중이므로, Playwright의 동기(Sync) API와 충돌함.  

**해결:** 코드를 async_playwright 기반의 비동기 방식으로 전면 수정하고, await 키워드를 적용함.  

---

### 문제 2: 브라우저 바이너리 및 OS 의존성 누락  
**현상:** 브라우저 실행 시 Executable doesn't exist 및 libnspr4.so: cannot open shared object file 에러 발생.  

**원인:** Playwright 라이브러리만 설치되고 실제 구동에 필요한 Chromium 브라우저 바이너리와 Linux OS 레벨의 C++ 공유 라이브러리가 설치되지 않음.  

**해결:** !playwright install chromium 및 !playwright install-deps (또는 sudo apt-get install) 명령어를 통해 필수 의존성 패키지를 설치함.  

---

## 2. 웹페이지 로딩 및 네트워크 제어 이슈

### 문제 3: 타임아웃 및 무한 로딩 대기  
**현상:** 페이지 접속 시점(page.goto)에서 진행되지 않고 에러 발생.  

**원인:** 대기 조건을 networkidle로 설정했으나, 사이트 내 백그라운드 트래커나 지속적인 네트워크 요청으로 인해 유휴 상태에 도달하지 못함.  

**해결:** 대기 조건을 wait_until="domcontentloaded"로 완화하고, 타겟 요소(text=TOP 10)가 나타날 때까지 명시적으로 기다리는 wait_for_selector를 추가하여 안정성을 확보함. 더불어 의심을 살 수 있는 구형 User-Agent 강제 주입 설정을 제거함.  

---

## 3. 동적 DOM 구조 및 데이터 탐색 이슈

### 문제 4: 커스텀 웹 컴포넌트(Shadow DOM 유사) 인식 불가  
**현상:** 일반적인 <ul>, <li> 태그 탐색 시 0개의 탭을 발견함.  

**원인:** 쉐보레 홈페이지가 <adv-grid>, <adv-col>, <gb-expander> 등 복잡하게 중첩된 커스텀 엘리먼트를 사용하고 있었음.  

**해결:** 개발자 도구 분석을 통해 실제 사용된 커스텀 태그를 CSS 선택자로 타겟팅하도록 코드를 전면 수정함.  

---

### 문제 5: 탭 전환 시 Stale Element Reference 오류  
**현상:** 두 번째 탭부터 요소를 찾지 못하고 Timeout 30000ms exceeded 에러 발생.  

**원인:** 인덱스(nth(i)) 기반으로 탭을 클릭하면, 클릭 후 DOM이 재렌더링되면서 기존에 메모리에 잡아두었던 요소의 연결이 끊어짐.  

**해결:** 페이지 진입 시 미리 전체 탭의 이름과 정보를 리스트(사전)에 텍스트 형태로 저장해 두고, 그 리스트를 순회하며 요소를 매번 새로 탐색(locator)하는 방식으로 변경함.  

---

## 4. 정밀 타겟팅 및 네비게이션 최적화 (최종)

### 문제 6: 이스케이프 문자 포함 텍스트 매칭 실패 및 불필요한 데이터 혼입  

**현상:**  

Android Auto/\nApple CarPlay와 같이 탭 이름에 공백이나 줄바꿈이 있어 텍스트 매칭(:has-text()) 클릭이 실패함.  

의도하지 않은 다른 영역의 아코디언 요소까지 수집 범위에 들어옴.  

**원인:** 텍스트 기반 클릭의 불안정성 및 부모 컨테이너 선택자의 모호함(adv-grid vs gb-adv-grid).  

**해결:**  

네비게이션 변경: 불안정한 탭 클릭(click()) 대신, 탭 요소의 href 속성값을 미리 추출하여 page.goto()를 통해 URL로 직접 이동하는 방식으로 변경하여 에러를 원천 차단함 (TOP 10 탭 제외 로직 포함).  

경로 정밀화: 개발자 도구를 통해 파악한 정확한 DOM 트리 경로(main#gb-main-content gb-adv-grid.gb-large-margin gb-expander)를 1:1로 적용하여 목표한 질문 데이터만 추출 성공.  

답변 추출 고도화: <gb-expander> 내부의 .gb-expander-btn을 클릭한 뒤, .gb-expander-content의 전체 inner_text()를 가져와 내부의 <p> 태그 데이터까지 깔끔하게 파싱함.  

---

## 💡 최종 결과

여러 차례의 DOM 분석과 로직 수정을 통해, 쉐보레 FAQ 페이지의 비표준적인 커스텀 엘리먼트 구조를 완벽하게 돌파했습니다. 수집된 JSON 데이터는 누락 없이 질문과 답변이 매핑되어 있으며, 이후 AI 에이전트의 RAG 데이터로 즉시 활용 가능한 고품질의 데이터셋으로 확보되었습니다.
